# 1. Base de conhecimento e recuperação

Evidências da execução real: fontes, notas, partições e calibração. O teste não é usado para escolher o limiar.

In [1]:
from pathlib import Path
import sys, json, hashlib
ROOT=Path.cwd()
if not (ROOT/'energia_clara').exists(): ROOT=ROOT.parent
assert (ROOT/'energia_clara').exists()
sys.path.insert(0,str(ROOT))
from energia_clara.config import read_json,read_jsonl
import pandas as pd
from IPython.display import display,Markdown
print('Python:',sys.version.split()[0])


Python: 3.14.7


In [2]:
manifest=read_json(ROOT/'avaliacao/manifesto_dados.json')
assert hashlib.sha256((ROOT/'avaliacao/teste.jsonl').read_bytes()).hexdigest()==manifest['teste_sha256']
display(pd.DataFrame([manifest]))
sources=read_json(ROOT/'base/fontes.json')
chunks=read_jsonl(ROOT/'base/trechos.jsonl')
display(pd.DataFrame([{'fonte':s['id'],'instituição':s['entidade'],'formato':s['formato'],'trechos':sum(c['fonte_id']==s['id'] for c in chunks),'url':s['url']} for s in sources]))
display(pd.DataFrame(chunks)[['id','titulo','texto','fonte_id','particao_qa']])

,seed,treino,validacao,teste,trechos,grupos_fatuais_teste,teste_sha256,separacao,preparado_antes_do_treino
0,42,200,20,60,30,10,5136b5ee1d2432e7e4ab455f6e9c8489e6e232f7c05542...,Grupos de fatos e perguntas disjuntos. As 4 fo...,True


,fonte,instituição,formato,trechos,url
0,S01,Cemig,HTML,3,https://www.cemig.com.br/faturas/entenda-sua-c...
1,S02,Inmetro,HTML,1,https://www.gov.br/inmetro/pt-br/assuntos/regu...
2,S03,ANEEL,HTML,3,https://www.gov.br/aneel/pt-br/assuntos/tarifa...
3,S04,ANEEL,HTML,1,https://www.gov.br/aneel/pt-br/assuntos/tarifa...
4,S05,Ministério de Minas e Energia,HTML,1,https://www.gov.br/mme/pt-br/acesso-a-informac...
5,S06,ANEEL,HTML,1,https://www.gov.br/aneel/pt-br/centrais-de-con...
6,S07,Cemig,HTML,8,https://www.cemig.com.br/noticia/dicas-e-orien...
7,S08,Cemig,HTML,5,https://www.cemig.com.br/noticia/dicas-e-orien...
8,S09,Cemig RI,HTML,3,https://ri.cemig.com.br/sustentabilidade/efici...
9,S10,Inmetro,HTML,3,https://www.gov.br/inmetro/pt-br/centrais-de-c...


,id,titulo,texto,fonte_id,particao_qa
0,E001,Cálculo do consumo,Consumo estimado em kWh = potência em W × hora...,S01,teste
1,E002,Tempo em minutos,"Para usar a fórmula de consumo, converta minut...",S01,teste
2,E003,Potência e energia,Watts (W) medem potência. Quilowatts-hora (kWh...,S01,teste
3,E004,Temperatura do ferro e tecido,Ajuste a temperatura do ferro ao tipo de tecid...,S08,treino
4,E005,Estimativa e fatura,Consumo em kWh multiplicado por uma tarifa inf...,S05,treino
5,E006,Valores de tarifa,Tarifas dependem da distribuidora e da vigênci...,S06,treino
6,E007,Tarifa convencional,"Na modalidade convencional, a tarifa de energi...",S03,treino
7,E008,Postos da Tarifa Branca,"Em dias úteis, a Tarifa Branca distingue ponta...",S03,treino
8,E009,Tarifa Branca no fim de semana,"Na Tarifa Branca, fins de semana e feriados na...",S03,treino
9,E010,Escolha da Tarifa Branca,A Tarifa Branca pode aumentar a conta quando o...,S04,treino


In [3]:
calibration=read_json(ROOT/'avaliacao/calibracao.json')
print('Limiar escolhido apenas na validação:',calibration['limiar'])
print('Recall@3 da validação:',calibration['recall_3'])
print('Rejeição correta na validação:',calibration['rejeicao_correta'])
display(pd.DataFrame(calibration['curva']).sort_values(['equilibrio','recall_3'],ascending=False).head(12))
display(pd.DataFrame([{'pergunta':q['pergunta'],'tipo':q['tipo'],'gold':q['trechos_relevantes'],'recuperados':q['recuperados']} for q in calibration['detalhes']]))

Limiar escolhido apenas na validação: 0.85
Recall@3 da validação: 1.0
Rejeição correta na validação: 0.75


,limiar,recall_3,rejeicao_correta,equilibrio
50,0.850,1.000000,0.750,0.875000
51,0.855,1.000000,0.750,0.875000
52,0.860,1.000000,0.750,0.875000
53,0.865,1.000000,0.750,0.875000
49,0.845,1.000000,0.625,0.812500
54,0.870,0.833333,0.750,0.791667
57,0.885,0.583333,1.000,0.791667
48,0.840,1.000000,0.500,0.750000
56,0.880,0.583333,0.875,0.729167
58,0.890,0.416667,1.000,0.708333


,pergunta,tipo,gold,recuperados
0,Qual é a localização exata do meu apartamento?,nao_consta,[],"[{'id': 'E022', 'score': 0.8392143249511719}, ..."
1,Qual a temperatura dentro da minha geladeira a...,nao_consta,[],"[{'id': 'E030', 'score': 0.8770797848701477}, ..."
2,Um aparelho em standby ainda pode consumir ene...,respondível,[E024],"[{'id': 'E024', 'score': 0.9009748697280884}, ..."
3,Qual é a leitura atual do meu medidor?,nao_consta,[],"[{'id': 'E014', 'score': 0.8234153389930725}, ..."
4,Como aproveitar a luz do dia para reduzir o ga...,respondível,[E017],"[{'id': 'E017', 'score': 0.878918468952179}, {..."
5,Que tipo de lâmpada a base recomenda para econ...,respondível,[E016],"[{'id': 'E016', 'score': 0.8676305413246155}, ..."
6,Quanto minha família pagou de energia no mês p...,nao_consta,[],"[{'id': 'E005', 'score': 0.8422410488128662}, ..."
7,"Em casa, como aproveitar a luz do dia para red...",respondível,[E017],"[{'id': 'E019', 'score': 0.8873354196548462}, ..."
8,"Em casa, que tipo de lâmpada a base recomenda ...",respondível,[E016],"[{'id': 'E016', 'score': 0.8670830130577087}, ..."
9,"Em casa, que alternativa ao ar-condicionado po...",respondível,[E029],"[{'id': 'E029', 'score': 0.8748351335525513}, ..."


O índice é local e exato, com vetores normalizados do E5. Recall é medido nas respondíveis; recusa é medida separadamente. As 40 formulações do teste pertencem a 10 grupos de fatos e não são independentes.